# SAM como referencia de comparación (baseline zero-shot)

La consigna (4.2) pide ejecutar SAM **sin entrenarlo**, con el **bounding box predicho** como prompt, sobre el conjunto de prueba, y reportar Dice e IoU frente al modelo propio. SAM no forma parte del pipeline ni genera etiquetas: es solo evidencia de cómo se comporta un modelo generalista fuera de dominio.

Este cuaderno es **autónomo**: carga el checkpoint del modelo propio (`best_C_conjunto.pth` o `modelo_multitarea_final.pt`, con o sin skip connections: detecta solo cuál es), predice las cajas con la cabeza grid y el NMS propio, y con **esas mismas cajas** corre SAM y el decoder propio. Usa `create_sam_predictor` y `run_sam_on_predicted_boxes` (secciones 23.12 y 23.13 de `proyecto_pengwin.ipynb`).

**Protocolo común (para que la comparación sea justa):**
- **Mismas cajas:** las de la cabeza grid propia tras el NMS, con la regla de `CONFIG_PIPELINE` de `pipeline_pengwin.py` (score mínimo 0.05, una caja por región) y el mismo cálculo que `inferir_cortes`. SAM recibe una caja por región como prompt; el decoder propio recibe esas mismas cajas como priors.
- **Mismos cortes:** los que tienen hueso en la referencia. Es la misma regla que se usa en las demás tablas de segmentación; los falsos positivos en cortes vacíos no se miden aquí.
- **Misma imagen:** ventana HU + Otsu/CLAHE + letterbox 256 × 256, preprocesada una sola vez. SAM la recibe repetida en 3 canales.
- **Misma métrica:** Dice e IoU **por región** (sacro, coxal izquierdo, coxal derecho), con intersección y áreas acumuladas sobre todos los cortes y casos (misma fórmula que 23.6b). SAM con una caja por región devuelve una región, no fragmentos, así que la comparación es a nivel de región; AP/PQ por fragmento no son comparables y no se reportan para SAM.
- Todo lo anterior queda guardado en `evidencias/sam/protocolo_<split>.json` junto a los resultados.

**Antes de correrlo:**
1. `pip install segment-anything` en el entorno del proyecto.
2. Descargar `sam_vit_b_01ec64.pth` (~375 MB) de https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth y dejarlo en `modelos_sam/`.
3. Agregar `modelos_sam/` al `.gitignore`: el checkpoint de SAM no se sube a GitHub.
4. Tener en `resultados/modelos/entrenamiento_multitarea/` el checkpoint del modelo que se quiere evaluar.


## 1. Configuración


In [1]:
import ast, gc, json, os, sys, time, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import SimpleITK as sitk
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from skimage.exposure import equalize_adapthist      # los usa enhance_bone_contrast (viene del cuaderno del proyecto)
from skimage.filters import threshold_otsu

warnings.filterwarnings("ignore", message="'mode' parameter is deprecated")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = False                              # igual que inferir_cortes (fp32): así las cajas coinciden exactamente con las del pipeline
IMAGE_SIZE = 256
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)


def find_project_root(marker="proyecto_pengwin.ipynb", max_up=3):
    start = Path(os.environ.get("PENGWIN_ROOT", Path.cwd())).resolve()
    for folder in [start, *list(start.parents)[:max_up]]:
        if (folder / marker).is_file():
            return folder
    raise FileNotFoundError(f"No encuentro {marker} desde {start}. Guarda este cuaderno en la carpeta del proyecto "
                            f"o escribe ROOT = Path(r'C:\\ruta\\del\\proyecto').")


ROOT = find_project_root()
BASE_DIR = ROOT                       # lo usa resolve_repo_path (viene de proyecto_pengwin.ipynb)
EVIDENCE_DIR = ROOT / "evidencias"
SAM_DIR = EVIDENCE_DIR / "sam"
SAM_DIR.mkdir(parents=True, exist_ok=True)
for _d in (ROOT / "src", ROOT):
    if _d.is_dir() and str(_d) not in sys.path:
        sys.path.insert(0, str(_d))


def _target_names(node):
    targets = node.targets if isinstance(node, ast.Assign) else [node.target] if isinstance(node, ast.AnnAssign) else []
    names = set()
    for target in targets:
        if isinstance(target, ast.Name):
            names.add(target.id)
    return names


def _load_existing_definitions(notebook_path, definition_names, value_names=()):
    notebook = json.loads(Path(notebook_path).read_text(encoding="utf-8"))
    wanted_defs, wanted_values = set(definition_names), set(value_names)
    snippets, found_defs, found_values = [], set(), set()
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        source = "".join(cell.get("source", []))
        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue
        for node in tree.body:
            name = getattr(node, "name", None)
            if isinstance(node, (ast.FunctionDef, ast.ClassDef)) and name in wanted_defs and name not in found_defs:
                snippets.append(ast.get_source_segment(source, node)); found_defs.add(name)
            elif isinstance(node, (ast.Assign, ast.AnnAssign)):
                selected = _target_names(node) & wanted_values
                if selected and not selected.issubset(found_values):
                    snippets.append(ast.get_source_segment(source, node)); found_values.update(selected)
    missing = (wanted_defs - found_defs) | (wanted_values - found_values)
    if missing:
        raise RuntimeError(f"Faltan definiciones en {Path(notebook_path).name}: {sorted(missing)}")
    exec("\n\n".join(snippets), globals())
    return len(snippets)


n_defs = _load_existing_definitions(
    ROOT / "proyecto_pengwin.ipynb",
    {"resolve_repo_path", "apply_hu_window", "enhance_bone_contrast", "instance_to_semantic", "letterbox_pair",
     "FundidoraPC", "FundidoraPCPlus", "build_backbone", "build_detection_region_priors", "SegmentationConvBlock",
     "backbone_feature_pyramid", "SkipFusionBlock", "CenterOffsetSkipDecoder",
     "create_sam_predictor", "run_sam_on_predicted_boxes"},
    {"CLASS_ID_TO_NAME", "TARGET_COLORS"})
try:
    from cbam_modulos import CBAM, CompuertaResidualCBAM
except ModuleNotFoundError:
        _load_existing_definitions(next((p for p in (ROOT / "cbam.ipynb", ROOT / "notebooks" / "cbam.ipynb") if p.is_file()), ROOT / "cbam.ipynb"), {"ChannelAttention", "SpatialAttention", "CBAM", "CompuertaResidualCBAM"})
from deteccion_grid import GridDetectionHead, NMS_CONFIG, nms_por_clase, postprocess
print(f"{n_defs} definiciones cargadas de proyecto_pengwin.ipynb | cbam | deteccion_grid.py | dispositivo: {DEVICE} (AMP {AMP})")

df = pd.read_csv(ROOT / "splits" / "dataset.csv", dtype={"case_id": str})
CASES = {s: sorted(df.loc[df["split"] == s, "case_id"]) for s in ("train", "val", "test")}
print({s: len(c) for s, c in CASES.items()})


17 definiciones cargadas de proyecto_pengwin.ipynb | cbam | deteccion_grid.py | dispositivo: cpu (AMP False)
{'train': 75, 'val': 5, 'test': 20}


In [2]:
SPLIT = "val"                      # "test" SOLO cuando el modelo y los parámetros estén congelados (la consigna pide evaluar test una vez)
SAM_MODEL_TYPE = "vit_b"           # vit_b es el más liviano; vit_l y vit_h necesitan más memoria
SAM_CHECKPOINT = ROOT / "modelos_sam" / "sam_vit_b_01ec64.pth"       # local; NO subir a Git (agregar modelos_sam/ al .gitignore)


ENTRENAMIENTO_DIR = ROOT / "resultados" / "modelos" / "entrenamiento_multitarea"
MODEL_PATH = ENTRENAMIENTO_DIR / "best_C_conjunto.pth"

# Regla de cajas del protocolo común: la MISMA que usa el pipeline de Fabián (`CONFIG_PIPELINE` en pipeline_pengwin.py).
BOX_SCORE_MIN, BOX_NMS = 0.05, dict(NMS_CONFIG)          # valores por defecto si no se puede importar el pipeline
try:
    from pipeline_pengwin import CONFIG_PIPELINE
    BOX_SCORE_MIN = CONFIG_PIPELINE["score_thresh"]
    BOX_NMS = {"iou_thresh": CONFIG_PIPELINE["iou_thresh"], "max_det_per_class": CONFIG_PIPELINE["max_det_per_class"]}
    BOX_RULE_SOURCE = "pipeline_pengwin.CONFIG_PIPELINE"
except Exception as exc:                                  # noqa: BLE001
    BOX_RULE_SOURCE = f"valores por defecto (no se pudo importar pipeline_pengwin: {type(exc).__name__})"

SLICE_STRIDE = 1                   # 2 = un corte de cada 2 (SAM en CPU es lento; en GPU no hace falta)
MAX_SLICES_PER_CASE = None         # p. ej. 150 para una prueba rápida
ALSO_GT_BOXES = False              # True: segunda corrida con cajas REALES (cota superior de SAM); no es el protocolo de la consigna
print(f"split={SPLIT} | casos: {CASES[SPLIT]} | modelo propio: {MODEL_PATH.name if MODEL_PATH else 'NO ENCONTRADO'} | "
      f"checkpoint SAM existe: {SAM_CHECKPOINT.is_file()}")
print(f"reglas de cajas ({BOX_RULE_SOURCE}): score >= {BOX_SCORE_MIN}, NMS {BOX_NMS}")


split=val | casos: ['035', '037', '067', '071', '079'] | modelo propio: best_C_conjunto.pth | checkpoint SAM existe: True
reglas de cajas (pipeline_pengwin.CONFIG_PIPELINE): score >= 0.05, NMS {'iou_thresh': 0.5, 'max_det_per_class': 1}


`SPLIT = "val"` sirve para probar y revisar. Para el número final, cambiar a `"test"` **una sola vez**, con el modelo y los parámetros congelados. En CPU, SAM puede tardar varios segundos por corte: `SLICE_STRIDE` y `MAX_SLICES_PER_CASE` permiten una prueba rápida o un subconjunto fijo (anotar cuál se usó para que Hoyos evalúe exactamente los mismos cortes).

**Verificación previa:** la celda siguiente revisa que estén la librería, el checkpoint de SAM y el del modelo propio.


In [3]:
problemas = []
try:
    import segment_anything                                                  
except ImportError:
    problemas.append("Falta la librería: en una terminal con el entorno del proyecto, `pip install segment-anything`.")
if not SAM_CHECKPOINT.is_file():
    problemas.append(f"Falta el checkpoint de SAM en {SAM_CHECKPOINT}. Descárgalo de "
                     "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth y guárdalo en modelos_sam/.")
elif SAM_CHECKPOINT.stat().st_size < 300e6:
    problemas.append(f"{SAM_CHECKPOINT.name} pesa {SAM_CHECKPOINT.stat().st_size / 1e6:.0f} MB; vit_b debe pesar ~375 MB (descarga incompleta).")
if MODEL_PATH is None:
    problemas.append(f"Falta el checkpoint del modelo propio en {ENTRENAMIENTO_DIR} (best_C_conjunto.pth o modelo_multitarea_final.pt).")
if not CASES[SPLIT]:
    problemas.append(f"splits/dataset.csv no tiene casos en el split '{SPLIT}'.")
print("Todo listo para correr SAM." if not problemas else "Antes de continuar:\n- " + "\n- ".join(problemas))


Todo listo para correr SAM.


## 2. Modelo propio y funciones

`load_own_model` reconstruye backbone, CBAM, cabeza grid y decoder desde el checkpoint. Si los pesos del decoder no coinciden con ninguna arquitectura conocida, falla con un mensaje claro en vez de cargar a medias. `predict_own_model` aplica la regla de cajas del protocolo y el decoder condicionado por esas cajas.


In [4]:
class CenterOffsetSegmentationDecoder(nn.Module):
    """Decoder SIN skips (solo f4 + priors), tal como lo entrenó Miguel en el checkpoint de la etapa C
    (versión del commit registrado en su config.json). Se incluye aquí para no depender de Git al cargar."""
    def __init__(self, in_channels=256):
        super().__init__()
        channels = (128, 96, 64, 32)
        blocks, current = [], in_channels
        for channel_count in channels:
            blocks.append(SegmentationConvBlock(current, channel_count))
            current = channel_count
        self.blocks = nn.ModuleList(blocks)
        self.semantic_head = nn.Conv2d(current + 3, 4, kernel_size=1)
        self.center_head = nn.Conv2d(current + 3, 3, kernel_size=1)
        self.offset_head = nn.Conv2d(current + 3, 2, kernel_size=1)

    def forward(self, feature_map, region_priors, output_size=(256, 256)):
        if tuple(region_priors.shape[-2:]) != tuple(output_size):
            region_priors = F.interpolate(region_priors.float(), size=output_size, mode="nearest")
        x = feature_map
        for block in self.blocks:
            x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
            x = block(x)
        x = F.interpolate(x, size=output_size, mode="bilinear", align_corners=False)
        x = torch.cat([x, region_priors.to(device=x.device, dtype=x.dtype)], dim=1)
        return {"semantic_logits": self.semantic_head(x), "center_logits": self.center_head(x),
                "offsets": torch.tanh(self.offset_head(x))}


class _SinSkips(nn.Module):
    """Adaptador: el decoder sin skips solo recibe f4; el resto del código le pasa (imagen, pirámide, priors)."""
    def __init__(self, decoder):
        super().__init__()
        self.decoder = decoder

    def forward(self, image, features, region_priors, output_size=(256, 256)):
        return self.decoder(features[-1], region_priors, output_size=output_size)


def _submodule_state(state, prefix):
    return {k[len(prefix):]: v for k, v in state.items() if k.startswith(prefix)}


def load_own_model(path, device=DEVICE):
    """Carga el modelo propio desde `modelo_multitarea_final.pt` (exportado) o `best_C_conjunto.pth` (checkpoint de la
    etapa C) y detecta solo si el decoder tiene skips o no. Devuelve (componentes, descripción)."""
    try:
        payload = torch.load(path, map_location="cpu", weights_only=True)
    except Exception:                 # checkpoint de entrenamiento: trae optimizador y config (archivo del propio equipo)
        payload = torch.load(path, map_location="cpu", weights_only=False)
    state = payload["model_state_dict"]
    arch = payload.get("architecture", {}) or {}
    cfg = payload.get("training_config", payload.get("config", {})) or {}
    backbone_name = arch.get("backbone_name", cfg.get("BACKBONE_NAME", "plus"))

    backbone = build_backbone(backbone_name, in_channels=1)
    backbone.load_state_dict(_submodule_state(state, "backbone."))
    has_cbam = any(k.startswith("cbam.") for k in state)
    cbam = None
    if has_cbam:
        cbam = CompuertaResidualCBAM(CBAM(256)) if "cbam.gamma" in state else CBAM(256)
        cbam.load_state_dict(_submodule_state(state, "cbam."))
    det_head = GridDetectionHead()
    det_head.load_state_dict(_submodule_state(state, "det_head."))

    seg_state, errors, decoder, kind = _submodule_state(state, "seg_decoder."), {}, None, None
    candidates = (("con skips", lambda: CenterOffsetSkipDecoder(in_channels=256, image_channels=1, use_skips=True)),
                  ("con skips (use_skips=False)", lambda: CenterOffsetSkipDecoder(in_channels=256, image_channels=1, use_skips=False)),
                  ("sin skips", lambda: _SinSkips(CenterOffsetSegmentationDecoder(in_channels=256))))
    for name, make in candidates:
        module = make()
        target = module.decoder if isinstance(module, _SinSkips) else module
        try:
            target.load_state_dict(seg_state)
        except RuntimeError as exc:
            errors[name] = str(exc).splitlines()[0][:160]
            continue
        decoder, kind = module, name
        break
    if decoder is None:
        raise RuntimeError("Los pesos del decoder no coinciden con ninguna arquitectura conocida:\n" +
                           "\n".join(f"  - {n}: {e}" for n, e in errors.items()) +
                           "\nSi cambió el decoder, usa la clase del commit del config.json (ver sección 24.0 de proyecto_pengwin.ipynb).")
    with_skips = kind.startswith("con skips")
    if with_skips:
        features_fn = backbone_feature_pyramid
    else:
        features_fn = lambda b, x: [None, None, None, b(x)[1]]     # el backbone de Miguel devuelve (pooled, f4)
    comp = {"backbone": backbone, "cbam": cbam, "det_head": det_head, "seg_decoder": decoder, "features_fn": features_fn}
    for module in (backbone, cbam, det_head, decoder):
        if module is not None:
            module.to(device).eval()
    selected = payload.get("selected_from", {"stage": payload.get("stage"), "epoch": payload.get("epoch")})
    stage = selected.get("stage")
    stage = stage.get("name") if isinstance(stage, dict) else stage
    info = {"archivo": Path(path).name, "backbone": backbone_name, "cbam": "compuerta" if "cbam.gamma" in state else ("CBAM" if has_cbam else "no"),
            "decoder": kind, "etapa": stage, "epoca": selected.get("epoch")}
    return comp, info


@torch.no_grad()
def predict_own_model(comp, images256, batch_size=8):
    """Imágenes 256 x 256 -> (cajas tras el NMS propio, semántica del decoder condicionado por esas cajas).
    Réplica de `inferir_cortes` (pipeline_pengwin.py): mismas cajas y el decoder corre en TODOS los cortes; si un corte
    no tiene cajas, sus priors son ceros. SAM, en cambio, no puede correr sin caja y deja ese corte vacío."""
    dets, sems = [], []
    for start in range(0, len(images256), batch_size):
        x = torch.from_numpy(np.stack(images256[start:start + batch_size])[:, None].astype(np.float32)).to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, enabled=AMP):
            feats = list(comp["features_fn"](comp["backbone"], x))
            if comp["cbam"] is not None:
                out = comp["cbam"](feats[-1])
                feats[-1] = out[0] if isinstance(out, tuple) else out
            det_out = comp["det_head"](feats[-1])
        detections = postprocess(det_out, score_thresh=BOX_SCORE_MIN, nms_fn=nms_por_clase, **BOX_NMS)
        priors = build_detection_region_priors(detections, output_size=(IMAGE_SIZE, IMAGE_SIZE), device=DEVICE)
        with torch.autocast(device_type=DEVICE.type, enabled=AMP):
            seg = comp["seg_decoder"](x, feats, priors)
        # mismo cálculo que decode_center_offset_instances: argmax de la probabilidad softmax (idéntico incluso en empates numéricos)
        sems.append(torch.softmax(seg["semantic_logits"].float(), dim=1).cpu().numpy().argmax(axis=1).astype(np.uint8))
        dets += [{k: d[k].detach().cpu().numpy() for k in ("boxes", "scores", "labels")} for d in detections]
    return dets, (np.concatenate(sems) if sems else np.zeros((0, IMAGE_SIZE, IMAGE_SIZE), np.uint8))


if MODEL_PATH is not None:
    own_model, own_info = load_own_model(MODEL_PATH)
    print("Modelo propio cargado:", own_info)
else:
    own_model, own_info = None, None
    print("AVISO: no hay checkpoint del modelo propio en", ENTRENAMIENTO_DIR, "- copia best_C_conjunto.pth (o modelo_multitarea_final.pt) ahí.")


Modelo propio cargado: {'archivo': 'best_C_conjunto.pth', 'backbone': 'plus', 'cbam': 'compuerta', 'decoder': 'con skips', 'etapa': 'C_conjunto', 'epoca': 7}


In [5]:
REGION_NAMES = {1: "sacro", 2: "coxal_izquierdo", 3: "coxal_derecho"}


class RegionAccumulator:
    """Dice e IoU por región acumulando intersección y áreas sobre todos los cortes (misma fórmula que 23.6b)."""
    def __init__(self):
        self.c = {k: [0.0, 0.0, 0.0] for k in (1, 2, 3)}      # inter, pred, gt

    def add(self, pred, gt):
        for k in (1, 2, 3):
            p, g = pred == k, gt == k
            self.c[k][0] += float((p & g).sum()); self.c[k][1] += float(p.sum()); self.c[k][2] += float(g.sum())

    def summary(self):
        out = {}
        for k, (i, p, g) in self.c.items():
            out[f"dice_{REGION_NAMES[k]}"] = 2 * i / (p + g) if p + g else np.nan
            out[f"iou_{REGION_NAMES[k]}"] = i / (p + g - i) if p + g - i else np.nan
        out["dice_medio"] = float(np.nanmean([out[f"dice_{n}"] for n in REGION_NAMES.values()]))
        out["iou_medio"] = float(np.nanmean([out[f"iou_{n}"] for n in REGION_NAMES.values()]))
        return out


def load_case(case_id):
    row = df[df["case_id"] == case_id].iloc[0]
    image = sitk.ReadImage(str(resolve_repo_path(row["image_path"])))
    label = sitk.GetArrayFromImage(sitk.ReadImage(str(resolve_repo_path(row["label_path"])))).astype(np.int16)
    return sitk.GetArrayFromImage(image).astype(np.float32), label


def boxes_from_gt(sem256):
    boxes, labels = [], []
    for k in (1, 2, 3):
        ys, xs = np.nonzero(sem256 == k)
        if len(ys):
            boxes.append([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1]); labels.append(k)
    return {"boxes": np.asarray(boxes, np.float32).reshape(-1, 4), "labels": np.asarray(labels, np.int64),
            "scores": np.ones(len(labels), np.float32)}


def sam_semantic(image256, det, predictor):
    """SAM con las cajas dadas -> máscara semántica 256 x 256 (0 fondo, 1..3 región de la caja)."""
    semantic = np.zeros((IMAGE_SIZE, IMAGE_SIZE), np.uint8)
    if det is None or len(det["boxes"]) == 0:
        return semantic
    instances, classes, _ = run_sam_on_predicted_boxes(image256, det, predictor)
    for instance_id, class_id in classes.items():
        semantic[instances == instance_id] = class_id
    return semantic


def run_sam_case(case_id, predictor, box_source="pred", keep_examples=6):
    """Un caso: SAM y modelo propio con las MISMAS cajas, sobre los mismos cortes y la misma imagen 256 x 256."""
    volume, label = load_case(case_id)
    positives = np.flatnonzero((label > 0).any(axis=(1, 2)))[::SLICE_STRIDE]      # cortes con hueso en la referencia
    if MAX_SLICES_PER_CASE:
        positives = positives[np.linspace(0, len(positives) - 1, min(MAX_SLICES_PER_CASE, len(positives))).astype(int)]
    images, gts = [], []
    for z in positives:                                  # un solo preprocesamiento para los dos métodos
        image256, gt256, _ = letterbox_pair(enhance_bone_contrast(apply_hu_window(volume[z])), label[z], IMAGE_SIZE)
        images.append(image256.astype(np.float32)); gts.append(instance_to_semantic(gt256))
    del volume
    if box_source == "pred":
        if own_model is None:
            raise FileNotFoundError("No hay checkpoint del modelo propio (MODEL_PATH): sin él no hay cajas predichas.")
        dets, model_sems = predict_own_model(own_model, images)
    acc_sam, acc_model = RegionAccumulator(), RegionAccumulator()
    sam_volume = np.zeros((len(label), IMAGE_SIZE, IMAGE_SIZE), np.uint8)
    rows, examples, seconds = [], [], []
    for n, z in enumerate(positives):
        image256, gt_sem = images[n], gts[n]
        det = dets[n] if box_source == "pred" else boxes_from_gt(gt_sem)
        t0 = time.time()
        sam_sem = sam_semantic(image256, det, predictor)
        seconds.append(time.time() - t0)
        sam_volume[z] = sam_sem
        acc_sam.add(sam_sem, gt_sem)
        mod_sem = None
        if box_source == "pred":
            mod_sem = model_sems[n]
            acc_model.add(mod_sem, gt_sem)
        present = [k for k in (1, 2, 3) if (gt_sem == k).any()]
        dice = lambda p: float(np.mean([2 * ((p == k) & (gt_sem == k)).sum() / max(((p == k).sum() + (gt_sem == k).sum()), 1) for k in present]))
        rows.append({"caso": case_id, "corte": int(z), "n_cajas": 0 if det is None else len(det["boxes"]),
                     "dice_sam": dice(sam_sem), "dice_modelo": dice(mod_sem) if mod_sem is not None else np.nan})
        examples.append((rows[-1]["dice_sam"], z, image256.astype(np.float16), gt_sem, mod_sem, sam_sem, det))
    examples.sort(key=lambda e: e[0])
    kept = examples[:keep_examples] + examples[-3:]
    return {"sam": acc_sam, "modelo": acc_model, "slices": pd.DataFrame(rows), "sam_volume": sam_volume,
            "examples": kept, "seconds_per_slice": float(np.mean(seconds)) if seconds else np.nan}


def plot_examples(examples, title, path=None):
    fig, axes = plt.subplots(len(examples), 4, figsize=(11, 2.8 * len(examples)), squeeze=False)
    cmap = np.array([[0, 0, 0, 0], [*plt.matplotlib.colors.to_rgba(TARGET_COLORS[1], 0.7)],
                     [*plt.matplotlib.colors.to_rgba(TARGET_COLORS[2], 0.7)], [*plt.matplotlib.colors.to_rgba(TARGET_COLORS[3], 0.7)]])
    for r, (dice_sam, z, img, gt, mod, sam, det) in enumerate(examples):
        for c, (mask, name) in enumerate(((None, "cajas predichas"), (gt, "referencia"), (mod, "modelo propio"), (sam, f"SAM (Dice {dice_sam:.2f})"))):
            ax = axes[r, c]; ax.imshow(img.astype(np.float32), cmap="gray"); ax.axis("off")
            if mask is not None:
                ax.imshow(cmap[mask])
            if c == 0 and det is not None:
                for b in det["boxes"]:
                    ax.add_patch(plt.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec="yellow", lw=1.2))
            ax.set_title(f"{name} · corte {z}" if c == 0 else name, fontsize=8)
    fig.suptitle(title); fig.tight_layout()
    if path:
        fig.savefig(path, dpi=140)
    return fig


## 3. Ejecución: SAM vs modelo propio

La tabla compara las dos segmentaciones sobre los mismos cortes y con las mismas cajas. Si un corte no tiene ninguna caja, SAM no puede correr y queda vacío; el decoder propio sí corre (sin priors), igual que en `inferir_cortes`, así que sus números coinciden con los del pipeline. Se guardan las máscaras de SAM por caso (`evidencias/sam/sam_<split>_<caso>.npz`, matriz `[corte, 256, 256]` con 0 fondo y 1/2/3 región, más qué cortes se procesaron) para que Hoyos calcule las métricas finales, y el Dice por corte (`dice_por_corte_<split>.csv`).


In [7]:
predictor = create_sam_predictor(SAM_CHECKPOINT, SAM_MODEL_TYPE, DEVICE)       # falla con un mensaje claro si falta la librería o el checkpoint

# Protocolo común: se guarda JUNTO a los resultados para que Hoyos (y el informe) sepan exactamente qué se evaluó.
protocol = {
    "split": SPLIT, "casos": CASES[SPLIT], "fecha": time.strftime("%Y-%m-%d %H:%M"),
    "modelo_propio": own_info, "cajas": {"score_min": BOX_SCORE_MIN, "nms": BOX_NMS, "origen": "cabeza grid propia + NMS propio", "regla_tomada_de": BOX_RULE_SOURCE},
    "imagen": "ventana HU -> Otsu + CLAHE -> letterbox 256 x 256 (la misma para SAM y para el modelo propio); SAM la recibe en 3 canales",
    "cortes": "cortes con hueso en la referencia", "stride": SLICE_STRIDE, "max_cortes_por_caso": MAX_SLICES_PER_CASE,
    "sam": {"tipo": SAM_MODEL_TYPE, "checkpoint": SAM_CHECKPOINT.name, "prompt": "una caja por región tras el NMS", "multimask": "mayor predicted_iou"},
    "metrica": "Dice e IoU por región (sacro, coxal izq., coxal der.) con intersección y áreas acumuladas sobre todos los cortes y casos; rejilla 256 x 256",
    "dispositivo": DEVICE.type, "torch": torch.__version__,
}
(SAM_DIR / f"protocolo_{SPLIT}.json").write_text(json.dumps(protocol, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

results, all_slices, examples_all = {}, [], []
accumulators = {"SAM (cajas predichas)": RegionAccumulator(), "Modelo propio (mismas cajas)": RegionAccumulator()}
for case_id in CASES[SPLIT]:
    res = run_sam_case(case_id, predictor, "pred")
    results[case_id] = res
    for key, name in (("sam", "SAM (cajas predichas)"), ("modelo", "Modelo propio (mismas cajas)")):
        for k in (1, 2, 3):
            for j in range(3):
                accumulators[name].c[k][j] += res[key].c[k][j]
    all_slices.append(res["slices"]); examples_all += res["examples"]
    np.savez_compressed(SAM_DIR / f"sam_{SPLIT}_{case_id}.npz", sam_semantic=res["sam_volume"],
                        processed=(res["sam_volume"].reshape(len(res["sam_volume"]), -1).any(axis=1)))
    print(f"caso {case_id}: {len(res['slices'])} cortes | {res['seconds_per_slice']:.2f} s por corte con SAM ({DEVICE.type}) | "
          f"Dice SAM {res['sam'].summary()['dice_medio']:.3f} vs modelo {res['modelo'].summary()['dice_medio']:.3f}")

summary = pd.DataFrame({name: acc.summary() for name, acc in accumulators.items()}).T
summary.to_csv(SAM_DIR / f"comparacion_sam_vs_modelo_{SPLIT}.csv")
display(summary.round(3))
slices_df = pd.concat(all_slices, ignore_index=True)
slices_df.to_csv(SAM_DIR / f"dice_por_corte_{SPLIT}.csv", index=False)
print(f"Latencia SAM por corte: {np.mean([r['seconds_per_slice'] for r in results.values()]):.2f} s ({DEVICE.type}, {SAM_MODEL_TYPE})")


KeyboardInterrupt: 

## 4. Galería de los peores cortes de SAM

Sirve para el informe: cada caso fallido debe ir acompañado de una hipótesis de causa.


In [ ]:
worst = sorted(examples_all, key=lambda e: e[0])[:5]
plot_examples(worst, f"Los 5 cortes con peor Dice de SAM ({SPLIT})", SAM_DIR / f"peores_sam_{SPLIT}.png"); plt.show()
print("Hipótesis a contrastar al mirar la galería: (1) SAM segmenta TODA la región ósea dentro de la caja y no separa fragmentos; "
      "(2) el contraste Otsu+CLAHE de la imagen no es el de las fotos naturales con que se entrenó SAM; "
      "(3) cajas que cubren dos huesos (sacro y coxal se tocan en la articulación) producen máscaras que se invaden.")


## 5. Cota superior con cajas reales (opcional)

No es el protocolo de la consigna. Indica cuánto del error de SAM viene de las cajas mal predichas y cuánto de SAM mismo.


In [ ]:
if ALSO_GT_BOXES:
    gt_box_acc = RegionAccumulator()
    for case_id in CASES[SPLIT]:
        run = run_sam_case(case_id, predictor, "gt")
        for k in (1, 2, 3):
            for j in range(3):
                gt_box_acc.c[k][j] += run["sam"].c[k][j]
    display(pd.DataFrame({"SAM (cajas REALES, cota superior)": gt_box_acc.summary()}).T.round(3))
else:
    print("ALSO_GT_BOXES = False: no se calcula la cota con cajas reales.")


## 6. Qué reportar

- Tabla de la sección 3 (Dice e IoU por región, SAM vs modelo propio) y el JSON del protocolo.
- Latencia por corte de SAM (indicar CPU o GPU y el tipo `vit_b`).
- La galería de la sección 4, con una hipótesis de causa por caso.
- La discusión: SAM recibe una caja por región y no separa fragmentos, así que no puede igualar el nivel de instancia del modelo propio; la comparación relevante es la de región. Un SAM que se acerca al modelo propio en Dice de región no implica que sirva para fragmentos.
- Si se usó un subconjunto de cortes (`SLICE_STRIDE`, `MAX_SLICES_PER_CASE`), decirlo y acordarlo con Hoyos para que las demás métricas usen los mismos cortes.
